In [1]:
from nltk.translate import bleu_score

In [2]:
import nltk
print(nltk.__file__)

c:\Users\shubh\micromamba\envs\pytorch-env\lib\site-packages\nltk\__init__.py


In [3]:
import sacrebleu
print(sacrebleu.__file__)

c:\Users\shubh\micromamba\envs\pytorch-env\lib\site-packages\sacrebleu\__init__.py


In [4]:
import math
from collections import Counter
from typing import Optional
from collections.abc import Sequence, Callable
def _ngrams(tokens: Sequence[str], n: int):
    """
    Sequence[str] ->
    ["Alice", "Bob"]       # list of str → valid
    ("Alice", "Bob")       # tuple of str → valid
    "Alice"                # string itself → valid (since str is a sequence of characters)
    [1, 2, 3]              # invalid, not str
    """
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

In [5]:
_ngrams(["shubhajeet", "is", "good"], 2)

[('shubhajeet', 'is'), ('is', 'good')]

In [6]:
set(_ngrams("I love you", 3))

{(' ', 'l', 'o'),
 (' ', 'y', 'o'),
 ('I', ' ', 'l'),
 ('e', ' ', 'y'),
 ('l', 'o', 'v'),
 ('o', 'v', 'e'),
 ('v', 'e', ' '),
 ('y', 'o', 'u')}

In [24]:
def jaccard_coefficient(word1: str, word2: str):
    cnt1 = Counter(_ngrams(word1, 2))
    cnt2 = Counter(_ngrams(word2, 2))
    # print(cnt1); print(cnt2)
    # print(cnt1 & cnt2)
    # print(cnt1 | cnt2)
    return sum((cnt1 & cnt2).values()) / sum((cnt1 | cnt2).values())

In [25]:
jaccard_coefficient("i am a good boy", "i am a good student")

0.5238095238095238

In [ ]:
import warnings, sys
class Fraction:
    def __init__(self, num, den):
        self.numerator = num
        self.denominator = den
# ============================================================
# 1. BLEU  (Papineni et al., 2002)
# ============================================================
def modified_precision(references: list[Sequence[str]], hypothesis: Sequence[str], n: int):
    hyp_ngrams = Counter(_ngrams(hypothesis, n))
    if not hyp_ngrams:
        return 0, 0
    max_ref_counts = Counter()
    for ref in references:
        ref_ngrams = Counter(_ngrams(ref, n))
        for ng, cnt in ref_ngrams.items():
            max_ref_counts[ng] = max(max_ref_counts[ng], cnt)
    clipped = {ng: min(cnt, max_ref_counts[ng]) for ng, cnt in hyp_ngrams.items()}
    return sum(clipped.values()), sum(hyp_ngrams.values())


def brevity_penalty(references: list[Sequence[str]], hypothesis: Sequence[str]):
    hyp_len = len(hypothesis)
    ref_lens = [len(r) for r in references]
    closest_ref_len = min(ref_lens, key=lambda rl: (abs(rl - hyp_len), rl))
    if hyp_len == 0:
        return 0.0
    if hyp_len > closest_ref_len:
        return 1.0
    return math.exp(1 - closest_ref_len / hyp_len)

class SmoothingFunction:
    def method0(self, p_n, *args, **kwargs):
        """
        No smoothing.
        """
        p_n_new = []
        for i, p_i in enumerate(p_n):
            if p_i.numerator != 0:
                p_n_new.append(p_i)
            else:
                _msg = str(
                    "\nThe hypothesis contains 0 counts of {}-gram overlaps.\n"
                    "Therefore the BLEU score evaluates to 0, independently of\n"
                    "how many N-gram overlaps of lower order it contains.\n"
                    "Consider using lower n-gram order or use "
                    "SmoothingFunction()"
                ).format(i + 1)
                warnings.warn(_msg)
                # When numerator==0 where denonminator==0 or !=0, the result
                # for the precision score should be equal to 0 or undefined.
                # Due to BLEU geometric mean computation in logarithm space,
                # we we need to take the return sys.float_info.min such that
                # math.log(sys.float_info.min) returns a 0 precision score.
                p_n_new.append(sys.float_info.min)
        return p_n_new

    def method1(self, p_n, *args, **kwargs):
        return [
            (
                (p_i.numerator + self.epsilon) / p_i.denominator
                if p_i.numerator == 0
                else p_i
            )
            for p_i in p_n
        ]

    def method2(self, p_n, *args, **kwargs):
        return [
            (
                Fraction(p_n[i].numerator + 1, p_n[i].denominator + 1, _normalize=False)
                if i != 0
                else p_n[0]
            )
            for i in range(len(p_n))
        ]

    def method3(self, p_n, *args, **kwargs):
        incvnt = 1  # From the mteval-v13a.pl, it's referred to as k.
        for i, p_i in enumerate(p_n):
            if p_i.numerator == 0:
                p_n[i] = 1 / (2**incvnt * p_i.denominator)
                incvnt += 1
        return p_n

    def method4(self, p_n, references, hypothesis, hyp_len=None, *args, **kwargs):
        incvnt = 1
        hyp_len = hyp_len if hyp_len else len(hypothesis)
        for i, p_i in enumerate(p_n):
            if p_i.numerator == 0 and hyp_len > 1:
                # incvnt = i + 1 * self.k / math.log(
                #     hyp_len
                # )  # Note that this K is different from the K from NIST.
                # p_n[i] = incvnt / p_i.denominator\
                numerator = 1 / (2**incvnt * self.k / math.log(hyp_len))
                p_n[i] = numerator / p_i.denominator
                incvnt += 1
        return p_n


def sentence_bleu(
    references: list[Sequence[str]],
    hypothesis: Sequence[str],
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function = None,
):
    ps = []
    for i, _ in enumerate(weights, start=1):
        num, den = modified_precision(references, hypothesis, i)
        ps.append(Fraction(num, den))

    if smoothing_function is not None:
        precisions = smoothing_function(ps)
    else:
        if min(pi.numerator for pi in ps) == 0:
            return 0.0
        precisions = [pi.numerator / pi.denominator for pi in ps]
    s = sum(w * math.log(p) for w, p in zip(weights, precisions) if p > 0)
    bp = brevity_penalty(references, hypothesis)
    return bp * math.exp(s)


def corpus_bleu(
    list_of_references: list[list[Sequence[str]]],
    hypotheses: list[Sequence[str]],
    weights=(0.25, 0.25, 0.25, 0.25),
):
    ps = [Fraction(0, 0)] * len(weights)
    hyp_lengths, ref_lengths = 0, 0

    for refs, hyp in zip(list_of_references, hypotheses):
        for i in range(1, len(weights) + 1):
            num, den = modified_precision(refs, hyp, i)
            ps[i-1].numerator += num
            ps[i-1].denominator[i - 1] += den
        hyp_lengths += len(hyp)
        ref_lengths += min(refs, key=lambda r: (abs(len(r) - len(hyp)), len(r))).__len__()

    if min(pi.numerator for pi in ps) == 0:
        return 0.0
    precisions = [pi.numerator / pi.denominator for pi in ps]
    s = sum(w * math.log(p) for w, p in zip(weights, precisions))
    bp = 1.0 if hyp_lengths > ref_lengths else math.exp(1 - ref_lengths / hyp_lengths)
    return bp * math.exp(s)


# ============================================================
# 2. chrF  (Popovic, 2015) -- character n-gram F-score
# ===========================================================
def _char_ngrams(text: str, n: int):
    text = text.replace(" ", "")  # sacrebleu strips whitespace by default
    return [text[i:i + n] for i in range(len(text) - n + 1)]

def sentence_chrf(hypothesis: str, references: list[str], char_order: int = 6, beta: int = 2):
    reference = references[0]  # simplified: single-reference, as in sacrebleu's default path
    precisions, recalls = [], []
    for n in range(1, char_order + 1):
        hyp_ngrams = Counter(_char_ngrams(hypothesis, n))
        ref_ngrams = Counter(_char_ngrams(reference, n))
        overlap = sum((hyp_ngrams & ref_ngrams).values())
        hyp_total = sum(hyp_ngrams.values())
        ref_total = sum(ref_ngrams.values())
        precisions.append(overlap / hyp_total if hyp_total else 0.0)
        recalls.append(overlap / ref_total if ref_total else 0.0)
    avg_p = sum(precisions) / char_order
    avg_r = sum(recalls) / char_order
    if avg_p + avg_r == 0:
        return 0.0
    beta2 = beta ** 2
    return (1 + beta2) * avg_p * avg_r / (beta2 * avg_p + avg_r) * 100

# ============================================================
# 3. TER  (Snover et al., 2006) -- Translation Edit Rate
# ============================================================
def _levenshtein(a: Sequence[str], b: Sequence[str]):
    m, n = len(a), len(b)
    # print(m, n)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost,  # substitution
            )
    return dp[m][n]

def sentence_ter(hypothesis: Sequence[str], references: list[Sequence[str]]):
    best_edits = min(_levenshtein(hypothesis, ref) for ref in references)
    avg_ref_len = sum(len(r) for r in references) / len(references)
    if avg_ref_len == 0:
        return 0.0
    return best_edits / avg_ref_len

In [17]:
hyp = "the cat sat on the mat".split()
ref = "the cat is sitting on the mat".split()
print("sentence_bleu:", sentence_bleu([ref], hyp))
print("sentence_chrf:", sentence_chrf(" ".join(hyp), [" ".join(ref)]))
print("sentence_ter :", sentence_ter(hyp, [ref]))

sentence_bleu: 0.0
sentence_chrf: 46.950097468255066
sentence_ter : 0.2857142857142857


In [13]:
" ".join(hyp)

'the cat sat on the mat'

In [2]:
import torch
print(torch.__file__)

c:\Users\shubh\micromamba\envs\pytorch-env\lib\site-packages\torch\__init__.py


In [18]:
hypothesis = "the cat sat on the mat"
reference = "the cat is sitting on the mat"
n = 2
hyp_ngrams = Counter(_char_ngrams(hypothesis, n))
ref_ngrams = Counter(_char_ngrams(reference, n))
print(hyp_ngrams)
print(ref_ngrams)

Counter({'at': 3, 'th': 2, 'he': 2, 'ec': 1, 'ca': 1, 'ts': 1, 'sa': 1, 'to': 1, 'on': 1, 'nt': 1, 'em': 1, 'ma': 1})
Counter({'th': 2, 'he': 2, 'at': 2, 'ti': 2, 'ec': 1, 'ca': 1, 'is': 1, 'ss': 1, 'si': 1, 'it': 1, 'tt': 1, 'in': 1, 'ng': 1, 'go': 1, 'on': 1, 'nt': 1, 'em': 1, 'ma': 1})


In [19]:
hyp_ngrams & ref_ngrams

Counter({'th': 2,
         'he': 2,
         'at': 2,
         'ec': 1,
         'ca': 1,
         'on': 1,
         'nt': 1,
         'em': 1,
         'ma': 1})